In [15]:
# !pip install pandas pymongo

In [16]:
import pandas as pd
from pymongo import MongoClient
from pymongo.errors import OperationFailure, ServerSelectionTimeoutError

In [17]:
MONGO_CONFIG = {
    "user": "labuser",
    "password": "labpass",
    "host": "localhost",
    "port": 27017,
    "database": "pima_indians_diabetes"
}

RAW_DIR = '../raw/clean'
COLLECTION_NAME = 'diabetes'

In [18]:
class MongoDatabase:

    """Class responsible for MongoDB connection management."""

    def __init__(self, db_config):
        # store configuration fields such as host, port, user, password, database
        self.client = None
        self.db = None
        self.config = db_config

    def connect(self):
        if self.client is not None:
            return self.db
        
        try:
            # create a MongoClient only if it does not exist yet
            # build the URI with authSource=admin
            # set a small serverSelectionTimeoutMS for friendlier failures
            # select the target database
            # test the connection with ping

            uri = (f"mongodb://{self.config['user']}:{self.config['password']}@{self.config['host']}:{self.config['port']}/?"
                "authSource=admin")
            self.client = MongoClient(uri, serverSelectionTimeoutMS=1000)
            self.db = self.client[self.config['database']]
            self.client.admin.command('ping')
            return self.db
        
            # if authentication fails, raise a clear message mentioning
            # prj/docker-compose.yml and the docker exec mongosh command

        except OperationFailure as e:
            self.client = None
            raise OperationFailure(
                "MongoDB authentication failed.\n"
                "Check credentials in Final-Project/docker-compose.yml.\n"
                "You can test manually with:\n"
                "docker exec -it lab-mongo mongosh -u labuser -p labpass --authenticationDatabase admin", e)
            
        except ServerSelectionTimeoutError as e:
            self.client = None
            raise ServerSelectionTimeoutError(
                "Could not connect to MongoDB server. "
                "Check if the container/service is running and accessible.", e)

    def is_connected(self):
        # return True if the client is available, otherwise False
        return self.client is not None

    def get_collection(self, collection_name):
        # return the collection object from the selected database
        if self.db is None:
            raise Exception("Database not connected. Call connect() first.")
        return self.db[collection_name]

    def close(self):
        # close the client and reset internal references
        if self.client is not None:
            self.client.close()
        self.client = None
        self.db = None


In [19]:
class MongoExecutor:
    """Class responsible for CRUD operations on a MongoDB collection."""

    def __init__(self, database, collection_name):
        # keep a reference to the database object
        # obtain the collection from the database object
        self.db = database
        self.collection = self.db.get_collection(collection_name)

    def select(self, filter_query=None, projection=None, limit=10):
        # implement a simple find() wrapper
        cursor = self.collection.find(filter_query or {}, projection)
        if limit is not None:
            cursor = cursor.limit(limit)
        return list(cursor)
        
    def insert_one(self, document):
        # insert a single document
        return self.collection.insert_one(document)

    def insert_many(self, documents):
        # insert many documents for the ingestion step
        return self.collection.insert_many(documents)

    def update_one(self, filter_query, update_query):
        # update a single document
        return self.collection.update_one(filter_query, update_query)

    def delete_one(self, filter_query):
        # delete a single document
        return self.collection.delete_one(filter_query)
    
    def delete_many(self, filter_query=None):
        # delete many documents for ingestion step (avoid repeating documents)
        return self.collection.delete_many(filter_query or {})

    def count_documents(self, filter_query=None):
        # count documents, defaulting to all documents
        return self.collection.count_documents(filter_query or {})

In [20]:
def map_external_record(record, row_number):
    """Transform one CSV row into one MongoDB document."""

    def casting(value):
        try:
           return float(value)
        except (ValueError, TypeError):
            return str(value)
        
    document = {}

    for key, value in record.items():
        document[key] = casting(value)

    return document

In [21]:
def ingest_data(executor, csv_path, limit=None):
    """Read a CSV file and insert its mapped rows into MongoDB."""

    df = pd.read_csv(csv_path)
    if limit is not None:
        df = df.head(limit)
    
    documents = [map_external_record(row, index+1) for index, row in df.iterrows()]

    if documents:
        executor.insert_many(documents)

    return len(documents)

In [22]:
db = MongoDatabase(MONGO_CONFIG)
db.connect()

collection = db.get_collection(COLLECTION_NAME)
executor = MongoExecutor(db, COLLECTION_NAME)

db.client.admin.command('ping')
print("Successfully connected to MongoDB!")
print("Document count:", collection.count_documents({}))

Successfully connected to MongoDB!
Document count: 771


In [23]:
# flush out old documents
executor.delete_many({})

DeleteResult({'n': 771, 'ok': 1.0}, acknowledged=True)

In [24]:
# define path
csv_path = RAW_DIR + '/diabetes-clean.csv'

In [25]:
from pprint import pprint

In [26]:
# test ingestion with sample
print(ingest_data(executor, csv_path, 10))
print(executor.count_documents({}))
pprint(executor.select({}))

10
10
[{'Age': 50.0,
  'BMI Category': 'obese',
  'Blood Pressure': 72.0,
  'Body Mass Index': 33.6,
  'Care Path': 'high-risk',
  'Clinic Region': 'North',
  'Diabetes Pedigree Function': 0.627,
  'Glucose': 148.0,
  'Insulin': 126.0,
  'Outcome': 1.0,
  'Patient Segment': 'adult',
  'Pregnancies': 6.0,
  'Skin Thickness': 35.0,
  '_id': ObjectId('6a1cc04e46de6cf863d81c36')},
 {'Age': 31.0,
  'BMI Category': 'overweight',
  'Blood Pressure': 66.0,
  'Body Mass Index': 26.6,
  'Care Path': 'routine follow-up',
  'Clinic Region': 'South',
  'Diabetes Pedigree Function': 0.351,
  'Glucose': 85.0,
  'Insulin': 126.0,
  'Outcome': 0.0,
  'Patient Segment': 'senior',
  'Pregnancies': 1.0,
  'Skin Thickness': 29.0,
  '_id': ObjectId('6a1cc04e46de6cf863d81c37')},
 {'Age': 32.0,
  'BMI Category': 'normal',
  'Blood Pressure': 64.0,
  'Body Mass Index': 23.3,
  'Care Path': 'urgent monitoring',
  'Clinic Region': 'North',
  'Diabetes Pedigree Function': 0.672,
  'Glucose': 183.0,
  'Insulin': 1

In [27]:
executor.delete_many({})

# full ingestion
print(ingest_data(executor, csv_path))
print(executor.count_documents({}))
executor.select({},{'_id': 0, 'Pregnancies': 1, 'Care Path': 1})

768
768


[{'Pregnancies': 6.0, 'Care Path': 'high-risk'},
 {'Pregnancies': 1.0, 'Care Path': 'routine follow-up'},
 {'Pregnancies': 8.0, 'Care Path': 'urgent monitoring'},
 {'Pregnancies': 1.0, 'Care Path': 'routine'},
 {'Pregnancies': 0.0, 'Care Path': 'unknown'},
 {'Pregnancies': 5.0, 'Care Path': 'routine follow-up'},
 {'Pregnancies': 3.0, 'Care Path': 'high-risk'},
 {'Pregnancies': 10.0, 'Care Path': 'unknown'},
 {'Pregnancies': 2.0, 'Care Path': 'high-risk'},
 {'Pregnancies': 8.0, 'Care Path': 'high-risk'}]